In [ ]:
import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Porque aspiro a ser todo un hombre",
    "Porque exijo mis deberes antes que mis derechos",
    "Por convicción y no por circunstancia",
    "Para alcanzar las conquistas universales y ofrecerlas a mi pueblo",
    "Porque me duele la Patria en mis entrañas y aspiro a calmar sus dolencias",
    "Porque ardo en deseos de despertar al hermano dormido",
    "Para prender una antorcha en el altar de la Patria",
    "Porque me dignifico y siento el deber de dignificar a mi institución",
    "Porque mi respetada libertad de joven y estudiante me impone la razón de respetar este recinto",
    "Porque traduzco la tricromía de mi bandera como trabajo deber y honor"
]

print("Corpus:")
for i, doc in enumerate(documents):
    print(f"  D{i+1}: \"{doc}\"")

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

tokenized_docs = [preprocess(doc) for doc in documents]

print("Documentos tokenizados:")
for i, tokens in enumerate(tokenized_docs):
    print(f"  D{i+1}: {tokens}")

In [ ]:
model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=50,
    window=5,
    min_count=1,
    workers=4,
    epochs=200,
    seed=42
)

print(f"Modelo Word2Vec entrenado")
print(f"Vocabulario: {len(model.wv)} palabras")
print(f"Dimensión de vectores: {model.wv.vector_size}")
print(f"\nEjemplo — vector de 'patria' (primeros 10 valores):")
print(f"  {model.wv['patria'][:10].round(4)}")

In [ ]:
def document_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

doc_vectors = np.array([document_vector(tokens, model) for tokens in tokenized_docs])

sim_matrix = cosine_similarity(doc_vectors)

labels = [f"D{i+1}" for i in range(len(documents))]
print("Matriz de similitud coseno entre documentos:\n")

print(f"{'':>5}", end="")
for label in labels:
    print(f"{label:>8}", end="")
print()

for i, label in enumerate(labels):
    print(f"{label:>5}", end="")
    for j in range(len(labels)):
        print(f"{sim_matrix[i][j]:>8.4f}", end="")
    print()

In [ ]:
n = len(documents)
max_sim = -1
min_sim = 2
most_similar_pair = (0, 0)
most_dissimilar_pair = (0, 0)

for i in range(n):
    for j in range(i + 1, n):
        if sim_matrix[i][j] > max_sim:
            max_sim = sim_matrix[i][j]
            most_similar_pair = (i, j)
        if sim_matrix[i][j] < min_sim:
            min_sim = sim_matrix[i][j]
            most_dissimilar_pair = (i, j)

i, j = most_similar_pair
print("=" * 60)
print("DOCUMENTOS MÁS SIMILARES:")
print(f"  D{i+1}: \"{documents[i]}\"")
print(f"  D{j+1}: \"{documents[j]}\"")
print(f"  Similitud coseno: {max_sim:.4f}")

i, j = most_dissimilar_pair
print(f"\nDOCUMENTOS MÁS DISÍMILES:")
print(f"  D{i+1}: \"{documents[i]}\"")
print(f"  D{j+1}: \"{documents[j]}\"")
print(f"  Similitud coseno: {min_sim:.4f}")
print("=" * 60)